# Perde eğrisi (kontur) modeli: Colab'da eğitim

Bir kontur modelini ücretsiz bir T4 GPU'da eğitir. Veri bilgisayarda hazırlanıp Google
Drive'a yüklenir; checkpoint'ler her epoch Drive'a yazılır. Hangi tur eğitileceği ilk kod
hücresindeki ayarlarda: varsayılan 2. tur (`guitar_contour5.yaml`, frekans ekseninde
konvolüsyonlu kafa, yarım ton başına 5 bin). 1. tur için `ROUND = 1`.

**Başlamadan önce**

1. Bilgisayarda `scripts/pack_dataset.py` ile oluşan zip'i (2. tur:
   `guitarset_contour5.zip`) Drive'da `MyDrive/music_transcriber/` klasörüne yükle.
2. *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU* seç.
3. *Çalışma zamanı → Tümünü çalıştır*.

**Oturum koparsa** (ücretsiz Colab birkaç saatte bir kapatabilir, sekme kapanınca da durur):
yeniden *Tümünü çalıştır*. Eğitim Drive'daki `last.pt`'den kaldığı epoch'tan devam eder;
`best.pt`, en iyi skor ve erken durdurma sayacı korunur.

**Bittiğinde** Drive'daki `checkpoints/<RUN>/best.pt` dosyasını (2. tur:
`checkpoints/guitar_contour5/best.pt`) indir.

In [ ]:
# Settings: which round to train, where things live on Drive, which branch.
ROUND = 2  # 1: linear contour head (guitar_contour.yaml); 2: conv head, 5 bins/semitone
SUFFIX = "" if ROUND == 1 else "5"
CONFIG = f"ml/configs/guitar_contour{SUFFIX}.yaml"
DATA_NAME = f"guitarset_contour{SUFFIX}"  # the zip's name and its folder inside
RUN = f"guitar_contour{SUFFIX}"
DRIVE_DIR = "/content/drive/MyDrive/music_transcriber"
BRANCH = "feature/contour-model"
DATA_ZIP = f"{DRIVE_DIR}/{DATA_NAME}.zip"
CKPT_DIR = f"{DRIVE_DIR}/checkpoints/{RUN}"
DATA_DIR = "/content/data"  # local disk: much faster to read than Drive
REPO = "/content/ai_music_transcriber"
REPO_URL = "https://github.com/ibrhmgldmr3/ai_music_transcriber.git"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# The code: clone the branch once, pull its latest commits on later runs.
import os

if not os.path.exists(REPO):
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO}
else:
    !git -C {REPO} pull --ff-only
%cd {REPO}
!git log -1 --oneline

In [ ]:
# Colab already has PyTorch with CUDA; this adds the project and its light dependencies.
!pip install -q -e . mir_eval
import torch

print(torch.__version__, "CUDA:", torch.cuda.is_available())

In [ ]:
# The preprocessed tracks: unzip from Drive to the local disk (once per session).
import zipfile
from pathlib import Path

data = Path(DATA_DIR) / DATA_NAME
if not any(data.glob("*.npz")):
    with zipfile.ZipFile(DATA_ZIP) as archive:
        archive.extractall(DATA_DIR)
print(len(list(data.glob("*.npz"))), "tracks in", data)

In [ ]:
# Train, resuming from Drive's last.pt when an earlier session got that far.
last = Path(CKPT_DIR) / "last.pt"
resume = f"--resume {last}" if last.exists() else ""
print("Resuming from", last) if resume else print("Fresh run")
# An A100 trains faster than two loader workers unpack tracks (~0.23 s per batch of 16):
# give it up to 8, as its runtime has the cores. Other GPUs keep the config's 2.
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""
workers = min(8, os.cpu_count() or 2) if "A100" in gpu else 2
print(gpu, "->", workers, "loader workers")
paths = f"paths.processed_dir={data} paths.checkpoint_dir={CKPT_DIR}"
!python scripts/train.py --config {CONFIG} --set {paths} training.num_workers={workers} {resume}

In [ ]:
# Progress: contour and note F1 on the validation player, epoch by epoch.
import pandas as pd

history = pd.read_csv(f"{CKPT_DIR}/history.csv")
history.plot(x="epoch", y=["val_contour_f1", "val_frame_f1"], grid=True)
# val_contour_f1 is the best F1 over thresholds 0.1-0.5 (val_contour_threshold).
best = torch.load(f"{CKPT_DIR}/best.pt", map_location="cpu", weights_only=True)
scores = {k: round(v, 4) for k, v in best["metrics"].items() if k.startswith("val_")}
print("best.pt: epoch", best["epoch"], scores)
history.tail()